# CausalPype vs Raw DoWhy + EconML — Code Comparison

This notebook performs the **exact same causal analyses** using two approaches:

1. **Raw DoWhy-GCM + EconML** — direct API calls, manual result handling
2. **CausalPype** — composable task pipeline

Both use the Framingham Heart Study dataset and the same causal graph. The goal is to compare code verbosity and workflow ergonomics.

## Shared Setup (identical for both approaches)

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import networkx as nx
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, IterativeImputer

# Load and preprocess data
df = pd.read_csv("data/framingham.csv")
cat_cols = ["male", "education", "currentSmoker", "BPMeds",
            "prevalentStroke", "prevalentHyp", "diabetes"]
cont_cols = [c for c in df.columns if c not in cat_cols and c != "TenYearCHD"]

cat_imputer = SimpleImputer(strategy="most_frequent")
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

df_for_mice = df.copy()
for col in cat_cols:
    df_for_mice[col] = df_for_mice[col].astype(float)
mice = IterativeImputer(max_iter=20, random_state=42, sample_posterior=False)
imputed = pd.DataFrame(mice.fit_transform(df_for_mice), columns=df_for_mice.columns, index=df_for_mice.index)
for col in cont_cols:
    df[col] = imputed[col].values
for col in cat_cols:
    df[col] = df[col].astype(int).astype("category")

# Define causal graph
adj = {
    "age":            ["sysBP", "diaBP", "totChol", "glucose", "BMI", "heartRate", "TenYearCHD"],
    "male":           ["sysBP", "diaBP", "totChol", "BMI", "heartRate", "currentSmoker", "TenYearCHD"],
    "education":      ["currentSmoker"],
    "currentSmoker":  ["cigsPerDay"],
    "cigsPerDay":     ["heartRate", "TenYearCHD"],
    "BMI":            ["sysBP", "diaBP", "totChol", "glucose", "diabetes", "TenYearCHD"],
    "sysBP":          ["prevalentHyp", "TenYearCHD"],
    "diaBP":          ["prevalentHyp"],
    "prevalentHyp":   ["BPMeds"],
    "glucose":        ["diabetes"],
    "diabetes":       ["TenYearCHD"],
    "totChol":        ["TenYearCHD"],
    "heartRate":      ["TenYearCHD"],
    "prevalentStroke": ["TenYearCHD"],
}
G = nx.DiGraph(adj)

print(f"Data: {df.shape[0]} rows x {df.shape[1]} cols")
print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Data: 4240 rows x 16 cols
Graph: 16 nodes, 33 edges


## Approach 1: Raw DoWhy-GCM + EconML

In [2]:
import dowhy.gcm as gcm
from dowhy.gcm.validation import RejectionResult
from econml.dml import LinearDML
import numpy as np

# Build and fit the SCM
scm = gcm.InvertibleStructuralCausalModel(G)
gcm.auto.assign_causal_mechanisms(scm, df, quality=gcm.auto.AssignmentQuality.BETTER)
gcm.fit(scm, df)

# Validate
rejection, structure_details = gcm.refute_causal_structure(G, df)
model_rejection = gcm.refute_invertible_model(scm, df)

# Arrow Strength
strengths = gcm.arrow_strength(scm, target_node="TenYearCHD")

# Intrinsic Causal Influence
influences = gcm.intrinsic_causal_influence(scm, target_node="TenYearCHD")

# ATE
ate = gcm.average_causal_effect(
    scm,
    target_node="TenYearCHD",
    interventions_alternative={"currentSmoker": lambda x: 1},
    interventions_reference={"currentSmoker": lambda x: 0},
    num_samples_to_draw=2000,
)

# CATE — must manually compute adjustment set via do-surgery + d-separation
g_surgery = G.copy()
g_surgery.remove_edges_from(list(G.out_edges("currentSmoker")))
adjustment_set = nx.algorithms.d_separation.find_minimal_d_separator(
    g_surgery, {"currentSmoker"}, {"TenYearCHD"}
)
auto_confounders = [n for n in adjustment_set if n not in {"currentSmoker", "TenYearCHD"}]
T = df["currentSmoker"].values.ravel()
Y = df["TenYearCHD"].values.ravel()
X = df[["age", "male", "BMI", "sysBP"]].values
W = df[auto_confounders].values if auto_confounders else None
est = LinearDML()
est.fit(Y, T, X=X, W=W)
cate_effects = est.effect(X)

# Intervention
intervention_samples = gcm.interventional_samples(
    scm,
    interventions={"sysBP": lambda x: 120},
    num_samples_to_draw=2000,
)

# Counterfactual
cf_samples = gcm.counterfactual_samples(
    scm,
    interventions={"sysBP": lambda x: 120},
    observed_data=df,
)

# Fairness Audit — requires two counterfactual calls + manual disparity computation
cf_priv = gcm.counterfactual_samples(
    scm, interventions={"male": lambda x: 1}, observed_data=df,
)
cf_unpriv = gcm.counterfactual_samples(
    scm, interventions={"male": lambda x: 0}, observed_data=df,
)
disparity = float(np.mean(cf_priv["TenYearCHD"].values) - np.mean(cf_unpriv["TenYearCHD"].values))
individual_unfairness = cf_priv["TenYearCHD"].values - cf_unpriv["TenYearCHD"].values
priv_mask = df["male"] == 1
unpriv_mask = df["male"] == 0
obs_gap = float(df.loc[priv_mask, "TenYearCHD"].mean() - df.loc[unpriv_mask, "TenYearCHD"].mean())

# Sensitivity Analysis — must manually implement each refutation test
rng = np.random.default_rng(42)
num_simulations = 10

# Placebo: permute treatment, re-fit, re-estimate
placebo_effects = []
for _ in range(num_simulations):
    data_placebo = df.copy()
    data_placebo["currentSmoker"] = rng.permutation(data_placebo["currentSmoker"].values)
    scm_p = gcm.InvertibleStructuralCausalModel(G)
    gcm.auto.assign_causal_mechanisms(scm_p, data_placebo, quality=gcm.auto.AssignmentQuality.BETTER)
    gcm.fit(scm_p, data_placebo)
    placebo_effects.append(float(gcm.average_causal_effect(
        scm_p, target_node="TenYearCHD",
        interventions_alternative={"currentSmoker": lambda x: 1},
        interventions_reference={"currentSmoker": lambda x: 0},
        num_samples_to_draw=2000,
    )))

# Subset: re-estimate on random 80% subsets
subset_effects = []
for _ in range(num_simulations):
    data_sub = df.sample(frac=0.8, random_state=int(rng.integers(1e9)))
    scm_s = gcm.InvertibleStructuralCausalModel(G)
    gcm.auto.assign_causal_mechanisms(scm_s, data_sub, quality=gcm.auto.AssignmentQuality.BETTER)
    gcm.fit(scm_s, data_sub)
    subset_effects.append(float(gcm.average_causal_effect(
        scm_s, target_node="TenYearCHD",
        interventions_alternative={"currentSmoker": lambda x: 1},
        interventions_reference={"currentSmoker": lambda x: 0},
        num_samples_to_draw=2000,
    )))

# Random common cause: add random confounder to graph
rcc_effects = []
for _ in range(num_simulations):
    data_rcc = df.copy()
    data_rcc["_random_cause"] = rng.standard_normal(len(data_rcc))
    graph_rcc = G.copy()
    graph_rcc.add_edges_from([("_random_cause", "currentSmoker"), ("_random_cause", "TenYearCHD")])
    scm_r = gcm.InvertibleStructuralCausalModel(graph_rcc)
    gcm.auto.assign_causal_mechanisms(scm_r, data_rcc, quality=gcm.auto.AssignmentQuality.BETTER)
    gcm.fit(scm_r, data_rcc)
    rcc_effects.append(float(gcm.average_causal_effect(
        scm_r, target_node="TenYearCHD",
        interventions_alternative={"currentSmoker": lambda x: 1},
        interventions_reference={"currentSmoker": lambda x: 0},
        num_samples_to_draw=2000,
    )))

Fitting causal mechanism of node _random_cause: 100%|██████████| 17/17 [00:00<00:00, 48.24it/s]  


## Approach 2: CausalPype

In [3]:
import causalpype as cp

model = cp.CausalModel(adj)
model.fit(df)

results = model.run([
    cp.Validate(),
    cp.ArrowStrength(target="TenYearCHD"),
    cp.IntrinsicCausalInfluence(target="TenYearCHD"),
    cp.ATE(treatment="currentSmoker", outcome="TenYearCHD",
           treatment_value=1, control_value=0, num_samples=2000),
    cp.CATE(treatment="currentSmoker", outcome="TenYearCHD",
            effect_modifiers=["age", "male", "BMI", "sysBP"]),
    cp.Intervention(interventions={"sysBP": 120}, outcome="TenYearCHD",
                    num_samples=2000),
    cp.Counterfactual(interventions={"sysBP": 120}, outcome="TenYearCHD"),
    cp.FairnessAudit(protected_attribute="male", outcome="TenYearCHD",
                     privileged_value=1, unprivileged_value=0),
    cp.SensitivityAnalysis(treatment="currentSmoker", outcome="TenYearCHD",
                           treatment_value=1, control_value=0,
                           num_simulations=10, num_samples=2000),
])

Fitting causal mechanism of node _random_cause: 100%|██████████| 17/17 [00:00<00:00, 26.62it/s] 


In [9]:
for r in results:
    print(r.summary())
    print()

╭──────────── Validation ────────────╮
│                                    │
│  Result: ISSUES FOUND              │
│                                    │
│  Structure                         │
│   Passed                ✗          │
│   N Tests               45         │
│   Bonferroni Level      ↑ 0.0011   │
│                                    │
│  Model                             │
│   Passed                ✗          │
│   Result                REJECTED   │
│                                    │
╰────────────────────────────────────╯

╭──────────────────────────── Arrow Strength ─────────────────────────────╮
│                                                                         │
│  Estimate:                                                              │
│   Target                        TenYearCHD                              │
│   BMI -> TenYearCHD                ↑ 0.0011  ████████████████████       │
│   diabetes -> TenYearCHD           ↑ 0.0005  ████████                   │
│   totChol -> TenYearCHD            ↑ 0.0004  ███████                    │
│   male -> TenYearCHD              ↓ -0.0003  ██████                     │
│   age -> TenYearCHD                ↑ 0.0003  █████                      │
│   sysBP -> TenYearCHD              ↑ 0.0003  █████                      │
│   prevalentStroke -> TenYearCHD    ↑ 0.0003  █████                      │
│   heartRate -> TenYearCHD         ↓ -0.0002  ████                       │
│   cigsPerDay -> TenYearCHD        ↓ -0.0002  ████                       │
│                                                                         │
│  Raw Strengths                                                          │
│   ('BMI', 'TenYearCHD')                ↑ 0.0011  ████████████████████   │
│   ('diabetes', 'TenYearCHD')           ↑ 0.0005  ████████               │
│   ('totChol', 'TenYearCHD')            ↑ 0.0004  ███████                │
│   ('male', 'TenYearCHD')              ↓ -0.0003  ██████                 │
│   ('age', 'TenYearCHD')                ↑ 0.0003  █████                  │
│   ('sysBP', 'TenYearCHD')              ↑ 0.0003  █████                  │
│   ('prevalentStroke', 'TenYearCHD')    ↑ 0.0003  █████                  │
│   ('heartRate', 'TenYearCHD')         ↓ -0.0002  ████                   │
│   ('cigsPerDay', 'TenYearCHD')        ↓ -0.0002  ████                   │
│                                                                         │
╰─────────────────────────────────────────────────────────────────────────╯

╭──────────────── Intrinsic Causal Influence ────────────────╮
│                                                            │
│  Estimate:                                                 │
│   Target                        TenYearCHD                 │
│   Total Variance Explained      ↑ 0.1185                   │
│   TenYearCHD              ↑ 0.1183  ████████████████████   │
│   diabetes                ↑ 0.0001                         │
│   age                     ↑ 0.0001                         │
│   sysBP                   ↑ 0.0001                         │
│   prevalentStroke        ↓ -0.0000                         │
│   male                    ↑ 0.0000                         │
│   cigsPerDay              ↑ 0.0000                         │
│   BMI                    ↓ -0.0000                         │
│   currentSmoker          ↓ -0.0000                         │
│   totChol                ↓ -0.0000                         │
│   glucose                 ↑ 0.0000                         │
│   education              ↓ -0.0000                         │
│   heartRate               ↑ 0.0000                         │
│                                                            │
│  Normalized                                                │
│   TenYearCHD              ↑ 0.9985  ████████████████████   │
│   diabetes                ↑ 0.0006                         │
│   age                     ↑ 0.0005                         │
│   sysBP                   ↑ 0.0005                         │
│   prevalentStroke        ↓ -0.0001                         │
│   male                    ↑ 0.0000                         │
│   cigsPerDay              ↑ 0.0000                         │
│   BMI                    ↓ -0.0000                         │
│   currentSmoker          ↓ -0.0000                         │
│   totChol                ↓ -0.0000                         │
│   glucose                 ↑ 0.0000                         │
│   education              ↓ -0.0000                         │
│   heartRate               ↑ 0.0000                         │
│                                                            │
╰────────────────────────────────────────────────────────────╯

╭────────────────────── ATE ──────────────────────╮
│                                                 │
│  Estimate: ↓ -0.0055                            │
│   Treatment                     currentSmoker   │
│   Outcome                       TenYearCHD      │
│   Treatment Value               1               │
│   Control Value                 0               │
│   Num Samples                   2,000           │
│                                                 │
╰─────────────────────────────────────────────────╯

╭───────────────────────── CATE ──────────────────────────╮
│                                                         │
│  Estimate: ↑ 0.0438                                     │
│   Treatment                     currentSmoker           │
│   Outcome                       TenYearCHD              │
│   Effect Modifiers              age, male, BMI, sysBP   │
│   Method                        linear_dml              │
│   Mean Effect                   ↑ 0.0438                │
│   STD Effect                    ↑ 0.0301                │
│   Bounds                        ↓ -0.0298, ↑ 0.1527     │
│                                                         │
╰─────────────────────────────────────────────────────────╯

╭─────────────────────── Intervention ───────────────────────╮
│                                                            │
│  Estimate: ↑ 0.1345                                        │
│   Outcome                       TenYearCHD                 │
│   Mean                          ↑ 0.1345                   │
│   STD                           ↑ 0.3427                   │
│                                                            │
│  Interventions                                             │
│   sysBP                        120  ████████████████████   │
│                                                            │
╰────────────────────────────────────────────────────────────╯

╭────────────────────── Counterfactual ──────────────────────╮
│                                                            │
│  Estimate: ↑ 0.1493                                        │
│   N Units                       4,240                      │
│   Outcome                       TenYearCHD                 │
│   Factual Mean                  ↑ 0.1519                   │
│   Counterfactual Mean           ↑ 0.1493                   │
│   Mean Effect                   ↓ -0.0026                  │
│                                                            │
│  Interventions                                             │
│   sysBP                        120  ████████████████████   │
│                                                            │
╰────────────────────────────────────────────────────────────╯

╭─────────────── Fairness Audit ───────────────╮
│                                              │
│  Estimate: ↑ 0.0031                          │
│   Protected Attribute           male         │
│   Outcome                       TenYearCHD   │
│   Counterfactual Disparity      ↑ 0.0031     │
│   Observational Gap             ↑ 0.0641     │
│   Mean Individual Unfairness    ↑ 0.0031     │
│   Max Individual Unfairness     ↑ 1.0000     │
│   N Privileged                  1,820        │
│   N Unprivileged                2,420        │
│                                              │
╰──────────────────────────────────────────────╯

╭─────────────────── Sensitivity Analysis ───────────────────╮
│                                                            │
│  Result: SENSITIVE                                         │
│   Original ATE                  ↓ -0.0085                  │
│                                                            │
│  Placebo                                                   │
│   p_value                 ↑ 0.6000  ████████████████████   │
│   std_effect              ↑ 0.0169                         │
│   mean_effect             ↑ 0.0040                         │
│   passed                         ✗                         │
│                                                            │
│  Subset                                                    │
│   passed                         ✓  ████████████████████   │
│   fraction                ↑ 0.8000  ████████████████       │
│   std_effect              ↑ 0.0083                         │
│   mean_effect            ↓ -0.0022                         │
│                                                            │
│  Random Common Cause                                       │
│   passed                         ✓  ████████████████████   │
│   std_effect              ↑ 0.0129                         │
│   original_ate           ↓ -0.0085                         │
│   mean_effect             ↑ 0.0029                         │
│                                                            │
╰────────────────────────────────────────────────────────────╯

## Comparison

Counting only code lines (no blanks, no comments), with all parameters explicit on both sides:

| | Raw DoWhy + EconML | CausalPype |
|---|---|---|
| **Code lines** | 94 | 20 |
| **Imports** | 4 | 1 |
| **SCM setup + fit** | 3 | 2 |
| **9 analyses** | 87 | 17 |

**Where does the ~5× difference come from?**

1. **`lambda` wrappers** — DoWhy requires `interventions={"node": lambda x: value}` for every intervention; CausalPype accepts plain values
2. **Different function signatures** — each DoWhy analysis has its own function name and parameter convention; CausalPype uses a uniform task API
3. **Automatic confounder selection** — CATE in raw DoWhy requires 5 lines of manual do-surgery + d-separation; CausalPype computes this from the graph
4. **Composite tasks** — FairnessAudit in raw DoWhy requires two separate `counterfactual_samples` calls plus manual disparity math (8 lines); CausalPype wraps this as a single task
5. **Sensitivity Analysis** — raw DoWhy requires manually implementing each refutation loop (placebo, subset, random common cause) with repeated SCM rebuilding (~40 lines); CausalPype provides this as a single task

CausalPype does **not** introduce new causal methods — it wraps the same DoWhy-GCM and EconML calls behind a uniform task interface. The sensitivity analysis task is notable because DoWhy's built-in refuters work with the classic API, not the GCM API — CausalPype bridges this gap.